# 🧠🎯 Warm-start + Benchmark com PokerBench

> **ARQUIVADO / NÃO EXECUTAR.** A concordância com PokerBench não previu força no cross-play; esta receita permanece somente como registro histórico.

**Celula unica - Run all (GPU).** A rede primeiro **clona as decisoes de um solver** (joga GTO) e depois faz **self-play** pra refinar. Medimos a **concordancia com o solver** (benchmark GTO) antes/depois.

Reaproveita o **`encode()`** do self-play (via o parser testado `poker_arena.ml.pokerbench`) - entao os pesos do warm-start transferem pro treino.

Registro histórico não executável; nenhuma credencial deve ser configurada.

In [ ]:
# ============================================================
# WARM-START + BENCHMARK com PokerBench (6-max NLHE, decisoes de solver/GTO)
# Registro historico bloqueado; sem clone ou publicacao.
# Celula unica: Run all. Roda em GPU.
# ============================================================
import os
import re
import subprocess
import sys
ARCHIVED_NOTEBOOK = True
PUBLISH_ARTIFACTS = os.environ.get('POKER_PUBLISH_ARTIFACTS') == '1'
raise RuntimeError('ARQUIVADO: warm-start PokerBench reprovado no cross-play')
from google.colab import userdata
HF_REPO_ID = userdata.get('HF_REPO_ID')
HF_REPO_ID_PATTERN = r'[A-Za-z0-9](?:[A-Za-z0-9._-]{0,94}[A-Za-z0-9])?/[A-Za-z0-9](?:[A-Za-z0-9._-]{0,94}[A-Za-z0-9])?'
if (not isinstance(HF_REPO_ID, str) or not HF_REPO_ID.isascii()
        or len(HF_REPO_ID) > 96 or re.fullmatch(HF_REPO_ID_PATTERN, HF_REPO_ID) is None
        or '..' in HF_REPO_ID or '--' in HF_REPO_ID):
    raise RuntimeError('HF_REPO_ID deve ser ASCII owner/repo canonico.')
# Clone privado legado removido: use um checkout pinado fora deste notebook arquivado.
safe_env_names = ('PATH', 'HOME', 'LANG', 'LC_ALL', 'TMPDIR', 'SYSTEMROOT', 'WINDIR', 'USERPROFILE', 'TEMP', 'TMP')
pip_env = {name: os.environ[name] for name in safe_env_names if name in os.environ}
pip_env.update({'PIP_INDEX_URL': 'https://pypi.org/simple', 'PIP_CONFIG_FILE': os.devnull})
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'poker-arena/backend',
                'stable-baselines3==2.7.0', 'sb3-contrib==2.7.0',
                'gymnasium==1.2.1', 'onnx==1.18.0', 'pandas==2.3.1',
                'huggingface_hub==0.34.3'], check=True, env=pip_env)  # noqa: S603
del pip_env
print('Notebook arquivado; nenhum clone ou upload foi executado.')

import numpy as np, pandas as pd, torch, time, copy
import torch.nn.functional as F
import gymnasium as gym
from gymnasium import spaces
from sb3_contrib import MaskablePPO
from stable_baselines3.common.callbacks import BaseCallback
from huggingface_hub import HfApi, hf_hub_download
from poker_arena.ml import FEATURE_SIZE, N_ACTIONS, SelfPlayEnv
from poker_arena.ml.pokerbench import featurize
import stable_baselines3 as _sb3
import sb3_contrib as _sbc
print(f'versoes -> sb3={_sb3.__version__}  sb3-contrib={_sbc.__version__}  '
      f'gymnasium={gym.__version__}  torch={torch.__version__}')


def box(titulo, linhas):
    w = max([len(titulo)] + [len(s) for s in linhas]) if linhas else len(titulo)
    print('\n  +' + '-'*(w+2) + '+'); print('  | ' + titulo.ljust(w) + ' |')
    print('  +' + '-'*(w+2) + '+')
    for s in linhas: print('  | ' + s.ljust(w) + ' |')
    print('  +' + '-'*(w+2) + '+')


# -------------------- CONFIG (aumente para treinar mais) --------------------
N_BC = 120_000          # registros do PokerBench no warm-start
BC_EPOCHS = 10
SELFPLAY_STEPS = 1_000_000
BC_ANCHOR_BATCHES = 12  # ancora no solver durante o self-play (anti-esquecimento)

box('WARM-START + BENCHMARK com PokerBench', [
    'A rede primeiro CLONA as decisoes de um solver (joga GTO), depois faz',
    'self-play pra refinar. Medimos a concordancia com o solver antes e depois.',
    'O BC e so o ponto de partida; o self-play depois POLE o agente.',
])

# -------------------- 1. carrega o PokerBench do HF --------------------
hf_token = userdata.get('HF_TOKEN')
if not isinstance(hf_token, str) or not hf_token:
    raise RuntimeError('HF_TOKEN e obrigatorio somente para leitura pinada.')
api = HfApi(token=hf_token)
csvs = [f for f in api.list_repo_files('RZ412/PokerBench', repo_type='dataset') if f.endswith('.csv')]
def grab(kw, split):
    fn = next(f for f in csvs if kw in f.lower() and split in f.lower())
    return pd.read_csv(hf_hub_download('RZ412/PokerBench', fn, repo_type='dataset',
                                       token=hf_token))
train = pd.concat([grab('preflop','train'), grab('postflop','train')], ignore_index=True)
test  = pd.concat([grab('preflop','test'),  grab('postflop','test')],  ignore_index=True)
hf_token = ''
train = train.sample(min(N_BC, len(train)), random_state=0)
box('POKERBENCH CARREGADO', [f'treino (warm-start): {len(train):,} registros',
                             f'teste (benchmark)  : {len(test):,} registros'])

# -------------------- 2. parse -> features (REUSA o nosso encode) --------------------
def to_arrays(df):
    Xs, Ms, Ys = [], [], []
    for row_index, r in enumerate(df.to_dict('records')):
        try:
            f, m, y = featurize(r)
        except (KeyError, TypeError, ValueError) as exc:
            raise RuntimeError(f'registro PokerBench invalido no indice {row_index}') from exc
        Xs.append(f); Ms.append(m); Ys.append(y)
    return np.asarray(Xs, np.float32), np.asarray(Ms, bool), np.asarray(Ys, np.int64)
Xtr, Mtr, Ytr = to_arrays(train)
Xte, Mte, Yte = to_arrays(test)
box('FEATURES PRONTAS', [f'treino: {Xtr.shape}  teste: {Xte.shape}',
                         'mesma codificacao do self-play (encode reaproveitado -> pesos transferem)'])

# -------------------- 3. monta o PPO --------------------
class GymPokerEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.env = SelfPlayEnv(n_players=6)
        self.observation_space = spaces.Box(0.0, 5.0, (FEATURE_SIZE,), dtype=np.float32)
        self.action_space = spaces.Discrete(N_ACTIONS)
        self._m = [True]*N_ACTIONS
    def reset(self, *, seed=None, options=None):
        o, self._m = self.env.reset(); return np.asarray(o, np.float32), {}
    def step(self, a):
        o, self._m, r, d = self.env.step(int(a)); return np.asarray(o, np.float32), float(r), bool(d), False, {}
    def action_masks(self):
        return np.asarray(self._m, bool)

env = GymPokerEnv()
model = MaskablePPO('MlpPolicy', env, n_steps=2048, batch_size=256, ent_coef=0.005,
                    gamma=0.99, gae_lambda=0.95, policy_kwargs=dict(net_arch=[256,256]),
                    verbose=0, device='auto')

def export_onnx(path='poker_expert.onnx'):
    dev = model.device
    model.policy.to('cpu').eval()  # exporta da propria politica (deepcopy falha pos-treino)
    class _Onnx(torch.nn.Module):
        def __init__(s):
            super().__init__()
            s.fe, s.mlp, s.an = model.policy.features_extractor, model.policy.mlp_extractor, model.policy.action_net
        def forward(s, obs):
            f = s.fe(obs); lp, _ = s.mlp(f); return s.an(lp)
    torch.onnx.export(_Onnx(), torch.zeros(1, FEATURE_SIZE), path, input_names=['obs'],
                      output_names=['logits'], dynamic_axes={'obs': {0: 'batch'}, 'logits': {0: 'batch'}},
                      opset_version=17, dynamo=False)
    model.policy.to(dev).train()  # volta pro device de treino

def agreement(X, M, Y):
    if len(X) == 0: return 0.0
    a, _ = model.predict(X, action_masks=M, deterministic=True)
    return float((np.asarray(a) == Y).mean()) * 100.0

box('BENCHMARK ANTES (politica aleatoria, sem treino)',
    [f'concordancia com o solver: {agreement(Xte, Mte, Yte):.1f}%'])

# -------------------- 4. WARM-START: behavioral cloning na politica do PPO --------------------
device = model.device  # GPU no Colab -> os tensores do BC precisam ir junto
Xt = torch.tensor(Xtr, device=device)
Yt = torch.tensor(Ytr, device=device)
actor = list(model.policy.mlp_extractor.policy_net.parameters()) + list(model.policy.action_net.parameters())
opt = torch.optim.Adam(actor, lr=1e-3)
box('WARM-START - clonando as decisoes do solver...',
    ['a rede aprende a jogar GTO ANTES de jogar contra si mesma'])
B = 1024
for ep in range(BC_EPOCHS):
    perm = torch.randperm(len(Xt), device=device); tot = 0.0
    for i in range(0, len(Xt), B):
        idx = perm[i:i+B]
        feats = model.policy.extract_features(Xt[idx])
        logits = model.policy.action_net(model.policy.mlp_extractor.forward_actor(feats))
        loss = F.cross_entropy(logits, Yt[idx])
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()*len(idx)
    print(f'   epoca {ep+1:>2}/{BC_EPOCHS}  perda={tot/len(Xt):.3f}  concordancia(teste)={agreement(Xte,Mte,Yte):.1f}%')

box('BENCHMARK APOS WARM-START',
    [f'concordancia com o solver: {agreement(Xte, Mte, Yte):.1f}%', '(deve subir MUITO vs aleatorio)'])

# rede de seguranca: salva a versao GTO-limpa do warm-start ANTES de qualquer self-play
export_onnx('poker_expert_warmstart.onnx')
try:
    api.upload_file(path_or_fileobj='poker_expert_warmstart.onnx', path_in_repo='poker_expert_warmstart.onnx', repo_id=HF_REPO_ID, repo_type='model')
except Exception as exc:
    print(f'aviso: upload do warm-start falhou ({type(exc).__name__}); artefato local preservado')
box('MODELO DO WARM-START SALVO (rede de seguranca)',
    ['poker_expert_warmstart.onnx guardado no HF (a versao GTO-limpa, ~58%).',
     'Se o self-play nao superar, ESTE e o Expert mais seguro.'])

# -------------------- 5. self-play continua do warm-start --------------------
def opp(feats, mask):
    a, _ = model.predict(np.asarray(feats, np.float32), action_masks=np.asarray(mask, bool), deterministic=False)
    return int(a)
env.env.set_opponent(opp)

def export_onnx(path='poker_expert.onnx'):
    dev = model.device
    model.policy.to('cpu').eval()  # exporta da propria politica (deepcopy falha pos-treino)
    class _Onnx(torch.nn.Module):
        def __init__(s):
            super().__init__()
            s.fe, s.mlp, s.an = model.policy.features_extractor, model.policy.mlp_extractor, model.policy.action_net
        def forward(s, obs):
            f = s.fe(obs); lp, _ = s.mlp(f); return s.an(lp)
    torch.onnx.export(_Onnx(), torch.zeros(1, FEATURE_SIZE), path, input_names=['obs'],
                      output_names=['logits'], dynamic_axes={'obs': {0: 'batch'}, 'logits': {0: 'batch'}},
                      opset_version=17, dynamo=False)
    model.policy.to(dev).train()  # volta pro device de treino

def barra(frac, n=22):
    frac = max(0.0, min(1.0, frac))
    cheio = int(frac * n)
    return '[' + '#' * cheio + '.' * (n - cheio) + f'] {100*frac:3.0f}%'


class Observa(BaseCallback):
    # painel Head First + LOGS AO VIVO no HF (resiliente: ve de qualquer lugar)
    def __init__(self, total, log_every=20_000, ckpt_every=250_000):
        super().__init__()
        self.total, self.log_every, self.ckpt_every = total, log_every, ckpt_every
        self._last = 0; self._nck = 0; self._prev = None; self._t0 = None; self._rows = []
    def _on_training_start(self):
        self._t0 = time.time()
        api.create_repo(HF_REPO_ID, repo_type='model', exist_ok=True, private=True)
        box('PAINEL DE TREINO AO VIVO  (so acompanhar, nao precisa fazer nada)', [
            'A IA esta jogando milhoes de maos contra si mesma pra aprender.',
            'A cada 20 mil maos eu mostro um resumo AQUI e salvo no HF. Como ler:',
            '',
            'BARRA ...... quanto ja foi (enche da esquerda pra direita)',
            'faltam ..... tempo que ainda falta (vai diminuindo sozinho)',
            'maos/s ..... velocidade  ->  se for maior que 0, NAO travou!',
            'lucro/mao .. fichas que a IA ganha por mao  ->  SUBINDO = mais forte',
            'GTO ........ % de acerto vs o solver  ->  ALTO = jogando certo',
        ])
        box('NAO TEM PERGUNTA IDIOTA', [
            'P: O lucro/mao as vezes DESCE. A IA esta piorando?',
            'R: Nao! Ela testa jogadas arriscadas pra aprender. Sobe e desce, mas a',
            '   TENDENCIA geral sobe. E normal - faz parte do aprendizado.',
            '',
            'P: A A100 e potente, por que ainda demora?',
            'R: O gargalo aqui NAO e a GPU - e simular as maos (roda na CPU). E como',
            '   uma Ferrari no transito: a potencia nao adianta na fila de carros.',
        ])
        print(f'   >> logs ao vivo em https://huggingface.co/{HF_REPO_ID}  (abra STATUS.md / training_log.csv)')
    def _hf(self, n, pct, sps, eta, rew, ag):
        self._rows.append((n, round(pct, 1), round(sps), round(eta, 1), round(rew, 1), round(ag, 1), round(time.time() - self._t0)))
        lines = ['passos,pct,maos_por_s,eta_min,lucro_mao,gto_pct,segundos'] + [','.join(map(str, r)) for r in self._rows]
        open('training_log.csv', 'w').write('\n'.join(lines) + '\n')
        md = ['# Poker Arena - status do treino (ao vivo)', '',
              f'**Passo {n:,} de {self.total:,}**  ({pct:.0f}%)', '',
              f'`{barra(n / self.total)}`  -  faltam ~{eta:.0f} min  -  {sps:.0f} maos/s', '',
              '| metrica | valor |', '|---|---|',
              f'| lucro/mao | {rew:+.1f} fichas |',
              f'| acerto GTO (vs solver) | {ag:.0f}% |',
              f'| checkpoints salvos | {self._nck} |', '',
              '> Atualizado automaticamente pelo job. Passos avancando + ETA caindo = tudo certo.']
        open('STATUS.md', 'w').write('\n'.join(md))
        try:
            api.upload_file(path_or_fileobj='training_log.csv', path_in_repo='training_log.csv', repo_id=HF_REPO_ID, repo_type='model')
            api.upload_file(path_or_fileobj='STATUS.md', path_in_repo='STATUS.md', repo_id=HF_REPO_ID, repo_type='model')
        except Exception:
            print('   (aviso: upload do log pro HF falhou desta vez - segue treinando normal)')
    def _bc_anchor(self):
        if BC_ANCHOR_BATCHES <= 0:
            return
        self.model.policy.set_training_mode(True)
        for _ in range(BC_ANCHOR_BATCHES):
            idx = torch.randint(0, len(Xt), (1024,), device=device)
            feats = model.policy.extract_features(Xt[idx])
            logits = model.policy.action_net(model.policy.mlp_extractor.forward_actor(feats))
            loss = F.cross_entropy(logits, Yt[idx])
            opt.zero_grad(); loss.backward(); opt.step()
    def _on_step(self):
        n = self.num_timesteps
        if n - self._last >= self.log_every:
            self._last = n
            el = max(time.time() - self._t0, 1e-9); sps = n / el
            eta = (self.total - n) / sps / 60 if sps > 0 else 0.0
            buf = self.model.ep_info_buffer
            rew = float(np.mean([e['r'] for e in buf])) if buf else 0.0
            seta = 'comecando' if self._prev is None else (
                'SUBINDO (^)' if rew > self._prev + 0.5 else
                ('caindo (v)' if rew < self._prev - 0.5 else 'estavel (=)'))
            self._prev = rew
            try:
                ag = agreement(Xte, Mte, Yte)
            except Exception as exc:
                raise RuntimeError('avaliacao de agreement falhou; selecao interrompida') from exc
            self.model.policy.set_training_mode(True)
            print()
            print(f'  {barra(n / self.total)}   faltam ~{eta:.0f} min   ({sps:.0f} maos/s)')
            print(f'     lucro/mao: {rew:+.1f} fichas  [{seta}]      acerto GTO: {ag:.0f}%')
            self._hf(n, 100 * n / self.total, sps, eta, rew, ag)
            self._bc_anchor()  # puxa de volta pro solver (anti-esquecimento)
        if n // self.ckpt_every > self._nck:
            self._nck += 1
            export_onnx('poker_expert.onnx')
            try:
                api.upload_file(path_or_fileobj='poker_expert.onnx', path_in_repo='poker_expert.onnx', repo_id=HF_REPO_ID, repo_type='model')
            except Exception as exc:
                print(f'aviso: upload do checkpoint falhou ({type(exc).__name__}); artefato local preservado')
            box(f'CHECKPOINT!  a IA ja jogou {n:,} maos', [
                'Modelo salvo em poker_expert.onnx e ENVIADO pro HF.',
                'Da pra baixar e jogar contra ELE agora, mesmo sem terminar o treino!',
            ])
        return True

box('SELF-PLAY a partir do warm-start (observabilidade ao vivo)...',
    [f'{SELFPLAY_STEPS:,} passos. Passos avancam + ETA cai = NAO travou.',
     'lucro/mao subindo e concordancia alta = convergindo bem. Checkpoints salvam o ONNX no meio.'])
model.learn(total_timesteps=SELFPLAY_STEPS, callback=Observa(SELFPLAY_STEPS))
model.save('poker_ppo_warmstart')
box('BENCHMARK FINAL (apos self-play)',
    [f'concordancia com o solver: {agreement(Xte, Mte, Yte):.1f}%'])

# -------------------- 6. export ONNX + publica no HF --------------------
export_onnx('poker_expert.onnx')  # reusa a funcao checkpoint-safe (deepcopy -> CPU)
api.create_repo(HF_REPO_ID, repo_type='model', exist_ok=True, private=True)
api.upload_file(path_or_fileobj='poker_expert.onnx', path_in_repo='poker_expert.onnx',
                repo_id=HF_REPO_ID, repo_type='model')
box('PUBLICADO NO HF HUB', [f'modelo: {HF_REPO_ID}',
                            'A Fase 4 (MLBot no backend) baixa este poker_expert.onnx daqui.'])
